# Mixture-of-Experts And Conditional Computation

This notebook studies a small sparse Mixture-of-Experts (MoE) router. The goal is not to reproduce a full production MoE stack; it is to isolate the routing mechanics that make conditional computation useful and fragile. We focus on three claims drawn from the MoE line running through GShard, Switch Transformers, and Mixtral [@gshard_2020; @switch_transformers_2021; @mixtral_of_experts_2024].


## Motivation

MoE layers increase total parameter count by storing many expert feed-forward networks, but they try to keep per-token compute bounded by activating only a few experts per token. GShard made this conditional-computation pattern practical at large scale [@gshard_2020]. Switch simplified the routing rule to top-1 and emphasized stability plus auxiliary balancing losses [@switch_transformers_2021]. Mixtral revived the modern sparse-LLM version with top-2 routing in a strong open-weight model [@mixtral_of_experts_2024].

The useful question for a first-principles notebook is narrower: what exactly do top-k routing, capacity limits, and load-balancing objectives do?

## Hypothesis

If the router uses top-k sparse routing with explicit capacity limits, then we can measure both the intended compute saving and the failure mode where many tokens collapse onto one expert. A simple load-balancing auxiliary loss should reduce that collapse, even when one expert starts with a strong initial bias.

## Baseline

The baseline is a collapsed top-1 router: every token initially prefers expert `0`, and each expert has finite capacity. That baseline is intentionally pathological. It makes the overflow and under-utilization behavior easy to see before we add an auxiliary balancing term.

## Metric

We track four small diagnostics:

- **Accepted load** per expert: how many assignments fit under capacity.
- **Dropped assignments** per expert: how many routing attempts overflowed capacity.
- **Load CV^2**: squared coefficient of variation of expert loads, where `0` means perfectly balanced load.
- **Switch-style load-balancing loss**: `E * sum_e mean(p_e) * f_e`, where `E` is the number of experts, `mean(p_e)` is the average router probability for expert `e`, and `f_e` is the fraction of accepted assignments that landed on expert `e`.

For the auxiliary-loss toy, we also track average reward under a synthetic routing objective so we can see whether better balance destroys the task signal or only trims a small margin.

## Mathematical derivation

Let `z_t in R^E` be router logits for token `t` over `E` experts. The soft routing probabilities are

`p_t = softmax(z_t)`.

Top-k routing keeps only the `k` largest entries of `p_t`. If `S_t` is that selected set, the sparse mixture weight for expert `e` is

`w_{t,e} = p_{t,e} / sum_{j in S_t} p_{t,j}` for `e in S_t`, and `0` otherwise.

A capacity limit turns the sparse choice into a constrained assignment problem. For `T` tokens, `E` experts, top-k routing, and capacity factor `c`, we use the standard per-expert capacity heuristic

`C = ceil(c * T * k / E)`.

So MoE sparsity has two different accounting stories:

1. **Parameter count** grows with the number of experts because all expert weights exist in memory.
2. **Active FLOPs per token** grow only with `k`, because only `k` experts execute for that token.

For a two-layer expert MLP with hidden width `d_hidden` and model width `d_model`, one expert has about `2 d_model d_hidden` weights. With `E` experts, the MoE layer stores about `E * 2 d_model d_hidden` expert weights, but top-k routing activates only `k * 2 d_model d_hidden` of those weights per token. That is the central MoE bargain: more stored parameters than a dense FFN, but less active compute than evaluating all experts every time.

To diagnose collapse, we compare the accepted expert-load vector against uniform load. The auxiliary term used below is a simple differentiable importance penalty,

`L_aux = E * sum_e (mean(p_e) - 1/E)^2`,

while the post-routing diagnostic uses the Switch-style product of mean probability and actual load fraction [@switch_transformers_2021].

In [ ]:
from pathlib import Path
import sys

import torch

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
SRC_PATH = ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from llm_from_scratch.research.moe import (
    expert_capacity,
    importance_loss,
    load_balance_loss,
    load_coefficient_of_variation_squared,
    route_tokens,
)

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)


## PyTorch implementation

The implementation surface is intentionally small:

- `route_tokens()` performs top-k routing, renormalizes the selected weights, and enforces a hard capacity limit.
- `load_balance_loss()` and `load_coefficient_of_variation_squared()` summarize collapse after routing.
- `importance_loss()` is the differentiable auxiliary term used in the toy optimization below.

We first inspect a tiny hand-written routing example before moving to the collapse experiment.

In [ ]:
def moe_parameter_counts(*, d_model: int, d_hidden: int, num_experts: int, top_k: int) -> dict[str, int]:
    router_params = d_model * num_experts
    expert_params_per_expert = 2 * d_model * d_hidden
    total_expert_params = num_experts * expert_params_per_expert
    total_params = router_params + total_expert_params
    active_expert_params_per_token = top_k * expert_params_per_expert
    dense_ffn_flops_per_token = num_experts * 4 * d_model * d_hidden
    active_ffn_flops_per_token = top_k * 4 * d_model * d_hidden
    return {
        'router_params': router_params,
        'expert_params_per_expert': expert_params_per_expert,
        'total_expert_params': total_expert_params,
        'total_params': total_params,
        'active_expert_params_per_token': active_expert_params_per_token,
        'dense_ffn_flops_per_token': dense_ffn_flops_per_token,
        'active_ffn_flops_per_token': active_ffn_flops_per_token,
    }


def synthetic_balance_rewards(num_tokens: int = 12, num_experts: int = 4) -> torch.Tensor:
    reward = torch.full((num_tokens, num_experts), 0.94, dtype=torch.float64)
    reward[:, 0] = 1.00
    for token_idx in range(num_tokens):
        reward[token_idx, token_idx % num_experts] = 0.995
    reward += 0.0001 * torch.randn_like(reward)
    return reward


def optimize_router(
    reward: torch.Tensor,
    *,
    aux_weight: float,
    steps: int = 60,
    lr: float = 0.2,
    init_bias: float = 3.0,
) -> dict[str, object]:
    logits = torch.zeros_like(reward, requires_grad=True)
    with torch.no_grad():
        logits[:, 0] = init_bias

    optimizer = torch.optim.Adam([logits], lr=lr)
    for _ in range(steps):
        gate_probs = torch.softmax(logits, dim=-1)
        task_loss = -(gate_probs * reward).sum(dim=-1).mean()
        aux_loss = importance_loss(gate_probs)
        loss = task_loss + aux_weight * aux_loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        result = route_tokens(logits.detach(), top_k=1, capacity_factor=1.25)
        accepted = result.accepted_mask.squeeze(-1)
        chosen_experts = result.topk_indices.squeeze(-1)
        chosen_reward = reward[torch.arange(reward.shape[0])[accepted], chosen_experts[accepted]]
        return {
            'logits': logits.detach(),
            'result': result,
            'mean_soft_reward': float((result.gate_probs * reward).sum(dim=-1).mean()),
            'mean_chosen_reward': float(chosen_reward.mean()),
            'cv2': float(load_coefficient_of_variation_squared(result.load)),
            'lb_loss': float(load_balance_loss(result.gate_probs, result.load)),
        }


In [ ]:
sample_logits = torch.tensor(
    [
        [4.0, 1.0, -2.0],
        [0.5, 3.0, 2.0],
        [-1.0, 0.0, 5.0],
    ],
    dtype=torch.float64,
)

top1_result = route_tokens(sample_logits, top_k=1, capacity_factor=8.0)
top2_result = route_tokens(sample_logits, top_k=2, capacity_factor=8.0)

capacity_logits = torch.tensor(
    [
        [8.0, 0.0],
        [7.0, 0.0],
        [6.0, 0.0],
        [5.0, 0.0],
    ],
    dtype=torch.float64,
)
capacity_result = route_tokens(capacity_logits, top_k=1, capacity_factor=0.5)

print('top-1 routed experts:', top1_result.topk_indices.squeeze(-1).tolist())
print('top-2 routed experts:')
print(top2_result.topk_indices)
print('top-2 mixture weights:')
print(top2_result.combine_weights)
print()
print('capacity stress test')
print('  capacity:', capacity_result.capacity)
print('  attempted:', capacity_result.attempted.tolist())
print('  accepted load:', capacity_result.load.tolist())
print('  dropped:', capacity_result.dropped.tolist())


### Parameter count vs active FLOPs

This is the MoE distinction people often blur:

- **Parameters** count every expert because every expert weight tensor exists.
- **Active FLOPs** count only the experts that actually run for one token.

So a sparse MoE can be a *larger parameter model* than a dense FFN while still executing *less active compute per token* than evaluating all experts.

In [ ]:
counts = moe_parameter_counts(d_model=8, d_hidden=16, num_experts=4, top_k=2)
for key, value in counts.items():
    print(f'{key:>30}: {value}')

flop_ratio = counts['dense_ffn_flops_per_token'] / counts['active_ffn_flops_per_token']
print()
print(f'dense-evaluate-all-experts vs active-top-2 FLOPs ratio: {flop_ratio:.1f}x')


## Numerical checks

The notebook checks four things directly:

1. top-1 and top-2 routing produce the expected sparse shapes;
2. capacity limits drop overflow assignments instead of silently overfilling an expert;
3. collapsed routing has much worse load-balance diagnostics than the mitigated router;
4. total MoE parameter count is larger than the per-token active parameter/FLOP footprint.


In [ ]:
reward = synthetic_balance_rewards()
collapsed = optimize_router(reward, aux_weight=0.0)
mitigated = optimize_router(reward, aux_weight=0.02)

top1_expected = torch.tensor([0, 1, 2], dtype=torch.int64)
top2_expected = torch.tensor([[0, 1], [1, 2], [2, 1]], dtype=torch.int64)

torch.testing.assert_close(top1_result.topk_indices.squeeze(-1), top1_expected)
torch.testing.assert_close(top2_result.topk_indices, top2_expected)
torch.testing.assert_close(top2_result.combine_weights.sum(dim=-1), torch.ones(3, dtype=torch.float64))

assert top1_result.topk_indices.shape == (3, 1)
assert top2_result.topk_indices.shape == (3, 2)
assert capacity_result.capacity == expert_capacity(4, 2, top_k=1, capacity_factor=0.5)
torch.testing.assert_close(capacity_result.load + capacity_result.dropped, capacity_result.attempted)
assert int(capacity_result.load[0].item()) == 1
assert int(capacity_result.dropped[0].item()) == 3

assert collapsed['cv2'] > 2.5
assert mitigated['cv2'] < 0.1
assert collapsed['lb_loss'] > 3.5
assert mitigated['lb_loss'] < 1.1
assert mitigated['mean_chosen_reward'] > 0.99

assert counts['total_params'] > counts['active_expert_params_per_token']
assert counts['dense_ffn_flops_per_token'] == 2 * counts['active_ffn_flops_per_token']

print('All numerical checks passed.')


## Ablations

The main ablation is the auxiliary-loss toggle:

- **Collapsed router (`aux_weight = 0.0`)**: expert `0` attracts every token, capacity overflows, and most tokens are dropped.
- **Mitigated router (`aux_weight = 0.02`)**: load spreads evenly enough to fill all four experts without sacrificing much reward.

This is a small synthetic example, but it isolates the mechanism cleanly: the auxiliary term does not create new expert skill; it changes the routing incentives so unused capacity becomes reachable.

In [ ]:
rows = [
    ('collapsed, no aux', collapsed),
    ('mitigated, aux=0.02', mitigated),
]

for name, payload in rows:
    result = payload['result']
    accepted = int(result.accepted_mask.sum().item())
    print(f'{name:>22} | load={result.load.tolist()} | dropped={result.dropped.tolist()} | accepted={accepted:>2d} | cv^2={payload["cv2"]:>5.2f} | lb_loss={payload["lb_loss"]:>5.2f} | reward={payload["mean_chosen_reward"]:>7.4f}')

print() 
print('Collapsed top-1 assignments:', collapsed['result'].topk_indices.squeeze(-1).tolist())
print('Mitigated top-1 assignments:', mitigated['result'].topk_indices.squeeze(-1).tolist())


## Assumptions

- The router is studied in isolation from the expert networks themselves.
- The capacity rule uses a simple ceiling heuristic rather than a distributed systems implementation detail such as token shuffling across devices.
- The auxiliary-loss toy optimizes only router logits, not a full language-model objective.
- FLOP accounting is approximate and focuses on expert MLP multiplies, not kernel-launch or communication overhead.

## Risks

- A toy router can make MoE balancing look easier than it is in a full model, where expert specialization, communication, and optimization interact.
- The synthetic reward matrix is designed to expose collapse, so its exact thresholds should not be over-interpreted.
- Top-k selection is discrete; the auxiliary optimization here relies on a soft importance penalty rather than differentiating through the hard assignment itself.
- Sparse activation reduces active FLOPs, but real wall-clock speed can still be limited by routing overhead and device imbalance.

## Failure criteria

This notebook fails its purpose if any of the following happen:

- top-1 and top-2 routing are not both exercised;
- capacity overflow is not visible in a deterministic example;
- the notebook confuses total parameter count with per-token active FLOPs;
- the auxiliary-loss example does not materially improve load-balance diagnostics over the collapsed baseline.

## Exercises

Work the paired exercise sheet in `exercises/research/22_moe_conditional_compute.md`. Minimum questions to answer without looking at the solutions:

1. Derive the capacity rule for `T` tokens, `E` experts, and top-k routing.
2. Explain why parameter count and active FLOPs diverge in sparse MoE layers.
3. Describe one way top-2 routing can help compared with top-1 routing, and one way it can hurt.
4. Explain why a load-balancing auxiliary loss can improve throughput even if it slightly lowers the best-expert reward.
5. Give one reason a perfectly balanced router might still be a bad MoE router.

## References

- GShard introduced large-scale conditional computation with sparse expert routing [@gshard_2020].
- Switch Transformers emphasized top-1 routing and auxiliary balancing for stability at scale [@switch_transformers_2021].
- Mixtral is a modern top-2 sparse MoE language model reference point [@mixtral_of_experts_2024].